In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTENC

import matplotlib.pyplot as plt

## Load the dataset
df = pd.read_csv('../data/merged.csv', sep=";")

df.describe()

,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
count,1044.000000,1044.000000,1044.000000,1044.000000,1044.000000,1044.000000,1044.000000,1044.000000,1044.000000,1044.000000,1044.000000,1044.000000,1044.000000,1044.000000,1044.000000,1044.000000
mean,16.726054,2.603448,2.387931,1.522989,1.970307,0.264368,3.935824,3.201149,3.156130,1.494253,2.284483,3.543103,4.434866,11.213602,11.246169,11.341954
std,1.239975,1.124907,1.099938,0.731727,0.834353,0.656142,0.933401,1.031507,1.152575,0.911714,1.285105,1.424703,6.210017,2.983394,3.285071,3.864796
min,15.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,16.000000,2.000000,1.000000,1.000000,1.000000,0.000000,4.000000,3.000000,2.000000,1.000000,1.000000,3.000000,0.000000,9.000000,9.000000,10.000000
50%,17.000000,3.000000,2.000000,1.000000,2.000000,0.000000,4.000000,3.000000,3.000000,1.000000,2.000000,4.000000,2.000000,11.000000,11.000000,11.000000
75%,18.000000,4.000000,3.000000,2.000000,2.000000,0.000000,5.000000,4.000000,4.000000,2.000000,3.000000,5.000000,6.000000,13.000000,13.000000,14.000000
max,22.000000,4.000000,4.000000,4.000000,4.000000,3.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,75.000000,19.000000,19.000000,20.000000


In [3]:
## Drop unnecessary features
df.drop(['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'higher', 'famrel', 'freetime', 'goout', 'health', 'activities', 'internet', 'romantic', 'famsup', 'nursery', 'paid'], axis=1, inplace=True)


## Convert the continuous grade value into categorical ranges
bins = [0, 9, 16, 20]
labels = ['At Risk', 'Satisfactory', 'Excellent']

df['risk_category'] = pd.cut(df['G3'], bins=bins, labels=labels, include_lowest=True)

## Change each category to a numerical value for the target feature and drop the columns
risk_cat_map = {
    'At Risk': 0,
    'Satisfactory': 1,
    'Excellent': 2
}

df['target'] = df['risk_category'].map(risk_cat_map)

df.drop(['G3', 'risk_category'], axis=1, inplace=True)

## Feature Engineering
df['grade_trajectory'] = df['G2'] - df['G1']
df['weighted_grade'] = 0.4*df['G1'] + 0.6*df['G2']
df['study_efficiency'] = df['G2'] / (df['studytime'] + 1)
df['absence_grade_interaction'] = df['absences']*(df['G1'] + df['G2'])

In [4]:
## Separate independent features from dependent variables
X = df.drop(['target'], axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

numerical_cols = ['age', 'G1', 'G2', 'failures', 'absences', 'grade_trajectory', 'weighted_grade', 'study_efficiency', 'absence_grade_interaction']
intensity_cols = ['Medu', 'Fedu', 'traveltime', 'studytime', 'Dalc', 'Walc']
categorical_cols = ['schoolsup']

## Build transformer pipelines
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])
intensity_transformer = Pipeline(steps=[
    ('minmax', MinMaxScaler())
])
categorical_transformer = Pipeline(steps=[
    ('encoder', OrdinalEncoder())
])

preprocessor = ColumnTransformer(transformers=[
    ('numerical', numerical_transformer, numerical_cols),
    ('intensity', intensity_transformer, intensity_cols),
    ('categorical', categorical_transformer, categorical_cols)
])

n_before_cat = len(numerical_cols) + len(intensity_cols)
categorical_col_indices = list(range(n_before_cat, n_before_cat + len(categorical_cols)))

In [6]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('smotenc', SMOTENC(categorical_features=categorical_col_indices, random_state=42)),
    ('classifier', GradientBoostingClassifier(learning_rate=0.05, max_depth=3, subsample=1, n_estimators=200))
])

model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('smotenc', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('intensity', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready 

In [ ]:
## Lowering the decision threshold for the At Risk class
y_prob = model.predict_proba(X_test)
print(y_prob)
class_order = model.classes_

at_risk_idx = list(class_order).index(0)

y_pred_adjusted = []

for prob in y_prob:
    if prob[at_risk_idx] >= 0.40:
        y_pred_adjusted.append(0)
    else:
        y_pred_adjusted.append(class_order[prob.argmax()])

## Evaluation
labels = ['At Risk', 'Satisfactory', 'Excellent']

print(classification_report(y_test, y_pred_adjusted, target_names=labels))

cm = confusion_matrix(y_test, y_pred_adjusted)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix Display - Student Performance Predictor')
plt.tight_layout()
plt.show()

ValueError: X does not contain any features, but ColumnTransformer is expecting 16 features

In [49]:
import pickle

with open('gradient_boosting', 'wb') as f:
    pickle.dump(model, f)

In [50]:
with open('gradient_boosting', 'rb') as f:
    model = pickle.load(f)

labels = ['At Risk', 'Satisfactory', 'Excellent']
pred = model.predict(X_test)
print(classification_report(y_test, pred, target_names=labels))

              precision    recall  f1-score   support

     At Risk       0.79      0.83      0.81        46
Satisfactory       0.94      0.93      0.93       149
   Excellent       0.93      0.93      0.93        14

    accuracy                           0.90       209
   macro avg       0.89      0.89      0.89       209
weighted avg       0.91      0.90      0.90       209

